In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!apt-get update -qq
!apt-get install -y -qq libxrender1 libxi6 libxkbcommon0 libsm6 libxxf86vm1 libgl1 colmap
!wget -q https://download.blender.org/release/Blender4.2/blender-4.2.1-linux-x64.tar.xz
!tar -xf blender-4.2.1-linux-x64.tar.xz
!./blender-4.2.1-linux-x64/blender --version
!colmap -h | head -3

In [ ]:
script = r'''
<paste the whole scene script here, unchanged>
'''
open('/kaggle/working/flight.py', 'w').write(script)

In [ ]:
%%writefile /kaggle/working/flight.py
import bpy, math, json, os

# ================= PARAMETERS =================
W, D, H  = 10.0, 8.0, 6.0
ROOF_H   = 3.0
OVERHANG = 0.6

N_VIEWS  = 20
RADIUS   = 28.0
ALTITUDE = 12.0
LENS_MM, SENSOR_MM = 24.0, 36.0
RES_X, RES_Y = 1024, 768
SAMPLES  = 48

OUT = '/kaggle/working/run_001'
PREVIEW_ONLY = False
# ==============================================

os.makedirs(OUT + '/images', exist_ok=True)

bpy.ops.object.select_all(action='SELECT')
bpy.ops.object.delete()

def noisy_material(name, scale, lo, hi, dark=False):
    mat = bpy.data.materials.new(name)
    mat.use_nodes = True
    nt = mat.node_tree
    bsdf = nt.nodes["Principled BSDF"]
    noise = nt.nodes.new("ShaderNodeTexNoise")
    noise.inputs['Scale'].default_value = scale
    noise.inputs['Detail'].default_value = 10.0
    noise.inputs['Roughness'].default_value = 0.7
    ramp = nt.nodes.new("ShaderNodeValToRGB")
    ramp.color_ramp.elements[0].position = lo
    ramp.color_ramp.elements[1].position = hi
    if dark:
        ramp.color_ramp.elements[0].color = (0.02, 0.02, 0.02, 1)
        ramp.color_ramp.elements[1].color = (0.30, 0.28, 0.26, 1)
    nt.links.new(noise.outputs['Fac'], ramp.inputs['Fac'])
    nt.links.new(ramp.outputs['Color'], bsdf.inputs['Base Color'])
    bsdf.inputs['Roughness'].default_value = 0.95
    return mat

concrete = noisy_material("concrete", 60.0, 0.35, 0.62)
ground   = noisy_material("ground",   25.0, 0.30, 0.70, dark=True)

bpy.ops.mesh.primitive_cube_add(size=1, location=(0, 0, H/2))
body = bpy.context.object
body.scale = (W, D, H)
body.data.materials.append(concrete)

roof_r = math.hypot(W, D)/2 + OVERHANG
bpy.ops.mesh.primitive_cone_add(vertices=4, radius1=roof_r, radius2=0,
                                depth=ROOF_H, location=(0, 0, H + ROOF_H/2),
                                rotation=(0, 0, math.pi/4))
bpy.context.object.data.materials.append(concrete)

bpy.ops.mesh.primitive_plane_add(size=RADIUS*6, location=(0, 0, 0))
bpy.context.object.data.materials.append(ground)

bpy.ops.object.light_add(type='SUN', location=(30, -30, 50))
bpy.context.object.data.energy = 4
bpy.context.object.rotation_euler = (0.6, 0.1, 0.9)

bpy.ops.object.empty_add(location=(0, 0, (H + ROOF_H)/2))
target = bpy.context.object

bpy.ops.object.camera_add()
cam = bpy.context.object
cam.data.lens = LENS_MM
cam.data.sensor_width = SENSOR_MM
bpy.context.scene.camera = cam
con = cam.constraints.new('TRACK_TO')
con.target = target
con.track_axis = 'TRACK_NEGATIVE_Z'
con.up_axis = 'UP_Y'

sc = bpy.context.scene
sc.render.engine = 'CYCLES'
sc.cycles.samples = SAMPLES
sc.cycles.device = 'CPU'
sc.render.resolution_x = RES_X
sc.render.resolution_y = RES_Y
prefs = bpy.context.preferences.addons['cycles'].preferences
prefs.compute_device_type = 'CUDA'   # 'OPTIX' also works on T4/P100
prefs.get_devices()
for d in prefs.devices:
    d.use = (d.type in {'CUDA', 'OPTIX'})
sc.cycles.device = 'GPU'

poses = []
count = 1 if PREVIEW_ONLY else N_VIEWS
for i in range(count):
    a = 2*math.pi*i/N_VIEWS
    cam.location = (RADIUS*math.cos(a), RADIUS*math.sin(a), ALTITUDE)
    bpy.context.view_layer.update()
    name = f'img_{i:03d}.png'
    sc.render.filepath = f'{OUT}/images/{name}'
    bpy.ops.render.render(write_still=True)
    poses.append({'file': name,
                  'location': list(cam.location),
                  'matrix_world': [list(r) for r in cam.matrix_world]})
    print(f'RENDERED {i+1}/{count}')

if not PREVIEW_ONLY:
    meta = {'params': {'W':W,'D':D,'H':H,'ROOF_H':ROOF_H,'N_VIEWS':N_VIEWS,
                       'RADIUS':RADIUS,'ALTITUDE':ALTITUDE,'LENS_MM':LENS_MM,
                       'SENSOR_MM':SENSOR_MM,'RES_X':RES_X,'RES_Y':RES_Y},
            'poses': poses}
    json.dump(meta, open(f'{OUT}/flight.json','w'), indent=2)
print('SCRIPT DONE')

In [ ]:
!./blender-4.2.1-linux-x64/blender --background --python /kaggle/working/flight.py 2>&1 | grep -E "RENDERED|SCRIPT DONE|Error|Traceback"

In [ ]:
!wc -l /kaggle/working/flight.py

In [ ]:
from PIL import Image
Image.open('/kaggle/working/run_001/images/img_000.png')

In [ ]:
!find /kaggle/input -maxdepth 3 | head -30

In [ ]:
!find /kaggle/input -maxdepth 3 | head -30

In [ ]:
!ls /kaggle/input/datasets/utkarshraj23/btp-run-001/

In [ ]:
!ls /kaggle/input/datasets/utkarshraj23/btp-run-001/btp-run-001/

In [ ]:
!mkdir -p /kaggle/working/run_001/images
!cp /kaggle/input/datasets/utkarshraj23/btp-run-001/btp-run-001/*.png /kaggle/working/run_001/images/
!ls /kaggle/working/run_001/images | wc -l

In [ ]:
!rm -rf /kaggle/working/blender-4.2.1-linux-x64 /kaggle/working/*.tar.xz
!rm -f /kaggle/working/run_001/db.db
!du -sh /kaggle/working

In [ ]:
%%writefile /kaggle/working/scene_only.py
import bpy, math, json, os

# ===== MUST MATCH the run that rendered the images =====
W, D, H  = 10.0, 8.0, 6.0
ROOF_H, OVERHANG = 3.0, 0.6
N_VIEWS, RADIUS, ALTITUDE = 20, 28.0, 12.0
LENS_MM, SENSOR_MM = 24.0, 36.0
RES_X, RES_Y = 1024, 768
OUT = '/kaggle/working/run_001/exports'
# =======================================================
os.makedirs(OUT, exist_ok=True)

bpy.ops.object.select_all(action='SELECT')
bpy.ops.object.delete()

bpy.ops.mesh.primitive_cube_add(size=1, location=(0, 0, H/2))
bpy.context.object.scale = (W, D, H)
roof_r = math.hypot(W, D)/2 + OVERHANG
bpy.ops.mesh.primitive_cone_add(vertices=4, radius1=roof_r, radius2=0, depth=ROOF_H,
                                location=(0, 0, H + ROOF_H/2), rotation=(0, 0, math.pi/4))

# ---- ground-truth mesh: structure only, no ground plane ----
bpy.ops.object.select_all(action='SELECT')
bpy.ops.wm.ply_export(filepath=f'{OUT}/gt_mesh.ply', export_selected_objects=True)

# ---- true camera poses, no rendering ----
bpy.ops.object.empty_add(location=(0, 0, (H + ROOF_H)/2))
target = bpy.context.object
bpy.ops.object.camera_add()
cam = bpy.context.object
cam.data.lens, cam.data.sensor_width = LENS_MM, SENSOR_MM
con = cam.constraints.new('TRACK_TO')
con.target, con.track_axis, con.up_axis = target, 'TRACK_NEGATIVE_Z', 'UP_Y'

poses = []
for i in range(N_VIEWS):
    a = 2*math.pi*i/N_VIEWS
    cam.location = (RADIUS*math.cos(a), RADIUS*math.sin(a), ALTITUDE)
    bpy.context.view_layer.update()
    poses.append({'file': f'img_{i:03d}.png',
                  'location': list(cam.matrix_world.translation),
                  'matrix_world': [list(r) for r in cam.matrix_world]})

json.dump({'params': dict(W=W, D=D, H=H, ROOF_H=ROOF_H, N_VIEWS=N_VIEWS, RADIUS=RADIUS,
                          ALTITUDE=ALTITUDE, LENS_MM=LENS_MM, SENSOR_MM=SENSOR_MM,
                          RES_X=RES_X, RES_Y=RES_Y),
           'poses': poses},
          open(f'{OUT}/true_poses.json', 'w'), indent=2)
print('GT MESH + POSES DONE')

import bmesh
bpy.context.view_layer.update()          # make sure transforms are applied
verts, faces = [], []
for o in bpy.data.objects:
    if o.type != 'MESH' or o.name.startswith('Plane'):
        continue
    bm = bmesh.new()
    bm.from_mesh(o.data)
    bm.transform(o.matrix_world)          # local -> world coordinates
    bmesh.ops.triangulate(bm, faces=bm.faces)
    base = len(verts)
    verts += [tuple(v.co) for v in bm.verts]
    faces += [tuple(base + v.index for v in f.verts) for f in bm.faces]
    bm.free()

with open(f'{OUT}/gt_mesh.ply', 'w') as f:
    f.write('ply\nformat ascii 1.0\n'
            f'element vertex {len(verts)}\n'
            'property float x\nproperty float y\nproperty float z\n'
            f'element face {len(faces)}\n'
            'property list uchar int vertex_indices\nend_header\n')
    for v in verts:
        f.write(f'{v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n')
    for t in faces:
        f.write(f'3 {t[0]} {t[1]} {t[2]}\n')
print(f'GT MESH: {len(verts)} verts, {len(faces)} tris')

In [ ]:
!wget -q https://download.blender.org/release/Blender4.2/blender-4.2.1-linux-x64.tar.xz -P /tmp
!tar -xf /tmp/blender-4.2.1-linux-x64.tar.xz -C /tmp
!/tmp/blender-4.2.1-linux-x64/blender --background --python /kaggle/working/scene_only.py 2>&1 | grep -E "DONE|Error|Traceback"

In [ ]:
!ls -la /kaggle/working/run_001/exports/

In [ ]:
!pip install -q trimesh
import trimesh
m = trimesh.load('/kaggle/working/run_001/exports/gt_mesh.ply')
print(m.vertices.shape, m.faces.shape, m.bounds)

In [ ]:
print(open('/kaggle/working/run_001/exports/gt_mesh.ply').read()[:600])

In [ ]:
!grep -n "primitive\|ply_export\|select_all" /kaggle/working/scene_only.py